#Realización de cubo y sabana

In [ ]:
import pandas as pd
import numpy as np

#Cargamos los archivos
orders = pd.read_csv("olist_orders_clean.csv")
items = pd.read_csv("olist_order_items_clean.csv")
payments = pd.read_csv("olist_order_payments_dataset_clean.csv")
reviews = pd.read_csv("olist_order_reviews_dataset_clean.csv")
products = pd.read_csv("olist_products_clean.csv")
customers = pd.read_csv("olist_customers_clean.csv")
geolocation = pd.read_csv("olist_geolocation_clean.csv")
sellers = pd.read_csv("olist_sellers_dataset_clean.csv")
category_translation = pd.read_csv("product_category_name_translation_clean.csv")

print("Archivos cargados correctamente:")
print("orders:", orders.shape)
print("items:", items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("customers:", customers.shape)
print("geolocation:", geolocation.shape)
print("sellers:", sellers.shape)
print("category_translation:", category_translation.shape)


# Normalizar nombres de columnas de status
orders["order_status"] = orders["order_status"].astype(str).str.strip().str.lower()

# Convertir fechas
columnas_fecha_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in columnas_fecha_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

# Normalizar ciudades a minúsculas para consistencia
customers["customer_city"] = customers["customer_city"].astype(str).str.strip().str.lower()
geolocation["geolocation_city"] = geolocation["geolocation_city"].astype(str).str.strip().str.lower()
sellers["seller_city"] = sellers["seller_city"].astype(str).str.strip().str.lower()


#Feature engineering en orders
orders["fecha"] = orders["order_purchase_timestamp"].dt.date
orders["anio"] = orders["order_purchase_timestamp"].dt.year
orders["mes"] = orders["order_purchase_timestamp"].dt.month
orders["dia"] = orders["order_purchase_timestamp"].dt.day
orders["trimestre"] = orders["order_purchase_timestamp"].dt.quarter
orders["dia_semana_num"] = orders["order_purchase_timestamp"].dt.dayofweek
orders["dia_semana_nombre"] = orders["order_purchase_timestamp"].dt.day_name()
orders["es_fin_semana"] = orders["dia_semana_num"].isin([5, 6]).astype(int)

# Métricas logísticas
orders["delivery_days_real"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_days_estimated"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["delivery_delay_days"] = orders["delivery_delay_days"].fillna(0)
orders["is_late_delivery"] = (orders["delivery_delay_days"] > 0).astype(int)
orders["is_delivered"] = (orders["order_status"] == "delivered").astype(int)

# Clave de fecha
orders["fecha_key"] = orders["order_purchase_timestamp"].dt.strftime("%Y%m%d")


#Feature engineering en products
products["product_volume_cm3"] = (
    products["product_length_cm"].fillna(0) *
    products["product_height_cm"].fillna(0) *
    products["product_width_cm"].fillna(0)
)

# Unir traducción de categorías
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

#Parte de geolocalización
def moda_segura(serie):
    moda = serie.mode()
    if len(moda) > 0:
        return moda.iloc[0]
    return serie.iloc[0] if len(serie) > 0 else np.nan

geolocation_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": moda_segura,
        "geolocation_state": moda_segura
    })
    .rename(columns={
        "geolocation_zip_code_prefix": "zip_code_prefix",
        "geolocation_lat": "geo_lat",
        "geolocation_lng": "geo_lng",
        "geolocation_city": "geo_city",
        "geolocation_state": "geo_state"
    })
)

#Resumen de pagos por pedido
payments_agg = (
    payments
    .groupby("order_id", as_index=False)
    .agg({
        "payment_value": "sum",
        "payment_installments": "max"
    })
)

payment_type_mode = (
    payments.groupby("order_id")["payment_type"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
    .reset_index()
)

payments_agg = payments_agg.merge(payment_type_mode, on="order_id", how="left")

#Resumen de reseñas por pedido
reviews_agg = (
    reviews
    .groupby("order_id", as_index=False)
    .agg({
        "review_score": "mean",
        "flag_short_message": "max",
        "flag_answer_before_creation": "max",
        "flag_review_id_duplicated": "max",
        "flag_review_id_multi_order": "max"
    })
)

reviews_agg["review_score"] = reviews_agg["review_score"].round(2)

#Dimensiones

dim_tiempo = (
    orders[[
        "fecha_key",
        "fecha",
        "anio",
        "mes",
        "dia",
        "trimestre",
        "dia_semana_num",
        "dia_semana_nombre",
        "es_fin_semana"
    ]]
    .drop_duplicates()
    .copy()
)

meses = {
    1: "enero", 2: "febrero", 3: "marzo", 4: "abril",
    5: "mayo", 6: "junio", 7: "julio", 8: "agosto",
    9: "septiembre", 10: "octubre", 11: "noviembre", 12: "diciembre"
}
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(meses)

# Reordenar columnas
dim_tiempo = dim_tiempo[[
    "fecha_key", "fecha", "anio", "mes", "nombre_mes", "trimestre",
    "dia", "dia_semana_num", "dia_semana_nombre", "es_fin_semana"
]]


dim_cliente = customers.copy()


dim_geografia_cliente = (
    customers[["customer_zip_code_prefix"]]
    .drop_duplicates()
    .rename(columns={"customer_zip_code_prefix": "zip_code_prefix"})
    .merge(geolocation_agg, on="zip_code_prefix", how="left")
)


dim_producto = products.copy()

dim_vendedor = sellers.copy()

# Agregar geolocalización de vendedor
geo_seller = geolocation_agg.rename(columns={
    "zip_code_prefix": "seller_zip_code_prefix",
    "geo_lat": "seller_geo_lat",
    "geo_lng": "seller_geo_lng",
    "geo_city": "seller_geo_city",
    "geo_state": "seller_geo_state"
})

dim_vendedor = dim_vendedor.merge(
    geo_seller,
    on="seller_zip_code_prefix",
    how="left"
)


dim_pago = (
    payments[["payment_type", "payment_installments"]]
    .drop_duplicates()
    .sort_values(["payment_type", "payment_installments"])
    .reset_index(drop=True)
)

dim_pago["pago_key"] = range(1, len(dim_pago) + 1)

# Vincular pago_key a payments_agg
payments_agg = payments_agg.merge(
    dim_pago,
    on=["payment_type", "payment_installments"],
    how="left"
)

# Reordenar columnas
dim_pago = dim_pago[["pago_key", "payment_type", "payment_installments"]]

#Preparamos tabla pase para fact
# Contar items por pedido para distribuir pagos
items_count = (
    items.groupby("order_id", as_index=False)
    .agg(num_items=("order_item_id", "count"))
)

# Tabla base
base = items.merge(
    orders,
    on="order_id",
    how="left"
)

base = base.merge(
    customers,
    on="customer_id",
    how="left"
)

base = base.merge(
    products,
    on="product_id",
    how="left"
)

base = base.merge(
    sellers,
    on="seller_id",
    how="left"
)

base = base.merge(
    items_count,
    on="order_id",
    how="left"
)

base = base.merge(
    payments_agg,
    on="order_id",
    how="left"
)

base = base.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

# Geolocalización cliente
base = base.merge(
    geolocation_agg.rename(columns={"zip_code_prefix": "customer_zip_code_prefix"}),
    on="customer_zip_code_prefix",
    how="left"
)

# Geolocalización vendedor
base = base.merge(
    geo_seller,
    on="seller_zip_code_prefix",
    how="left"
)

#Feature engineering en la tabla base
# Valor total del item
base["total_item_value"] = base["price"].fillna(0) + base["freight_value"].fillna(0)

# Pago asignado por item
base["payment_value_item"] = base["payment_value"] / base["num_items"]
base["payment_value_item"] = base["payment_value_item"].fillna(0)

# Indicadores de reseña
base["is_bad_review"] = (base["review_score"] <= 2).astype(int)
base["is_good_review"] = (base["review_score"] >= 4).astype(int)

#Tabla de fact
fact_ventas = base[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    "fecha_key",
    "pago_key",
    "order_status",
    "price",
    "freight_value",
    "total_item_value",
    "payment_value_item",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review",
    "customer_zip_code_prefix"
]].copy()

#sabana de ventas
tad_ventas = base[[
    "order_id",
    "order_item_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "geo_lat",
    "geo_lng",
    "geo_city",
    "geo_state",
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
    "seller_geo_lat",
    "seller_geo_lng",
    "payment_type",
    "payment_installments",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "anio",
    "mes",
    "dia",
    "trimestre",
    "dia_semana_num",
    "dia_semana_nombre",
    "es_fin_semana",
    "price",
    "freight_value",
    "total_item_value",
    "payment_value_item",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review"
]].copy()

#Base de pedidos
base_pedidos = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    items_count,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    payments_agg,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    reviews_agg,
    on="order_id",
    how="left"
)

base_pedidos = base_pedidos.merge(
    geolocation_agg.rename(columns={"zip_code_prefix": "customer_zip_code_prefix"}),
    on="customer_zip_code_prefix",
    how="left"
)


#Feature engineering para pedidos
base_pedidos["num_items"] = base_pedidos["num_items"].fillna(0).astype(int)
base_pedidos["tiene_items"] = (base_pedidos["num_items"] > 0).astype(int)

base_pedidos["is_bad_review"] = (base_pedidos["review_score"] <= 2).astype(int)
base_pedidos["is_good_review"] = (base_pedidos["review_score"] >= 4).astype(int)

#Fact pedidos
fact_pedidos = base_pedidos[[
    "order_id",
    "customer_id",
    "fecha_key",
    "pago_key",
    "order_status",
    "num_items",
    "tiene_items",
    "payment_value",
    "review_score",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review",
    "customer_zip_code_prefix"
]].copy()

#Sabana pedidos
tad_pedidos = base_pedidos[[
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "geo_lat",
    "geo_lng",
    "geo_city",
    "geo_state",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "fecha",
    "fecha_key",
    "anio",
    "mes",
    "dia",
    "trimestre",
    "dia_semana_num",
    "dia_semana_nombre",
    "es_fin_semana",
    "num_items",
    "tiene_items",
    "payment_type",
    "payment_installments",
    "payment_value",
    "review_score",
    "flag_short_message",
    "flag_answer_before_creation",
    "flag_review_id_duplicated",
    "flag_review_id_multi_order",
    "delivery_days_real",
    "delivery_days_estimated",
    "delivery_delay_days",
    "is_late_delivery",
    "is_delivered",
    "is_bad_review",
    "is_good_review"
]].copy()

#Exportamos resultados
dim_tiempo.to_csv("dim_tiempo.csv", index=False)
dim_cliente.to_csv("dim_cliente.csv", index=False)
dim_geografia_cliente.to_csv("dim_geografia_cliente.csv", index=False)
dim_producto.to_csv("dim_producto.csv", index=False)
dim_vendedor.to_csv("dim_vendedor.csv", index=False)
dim_pago.to_csv("dim_pago.csv", index=False)

fact_ventas.to_csv("fact_ventas.csv", index=False)
tad_ventas.to_csv("tad_ventas.csv", index=False)

fact_pedidos.to_csv("fact_pedidos.csv", index=False)
tad_pedidos.to_csv("tad_pedidos.csv", index=False)

print("\nETL finalizado correctamente.")
print("Archivos generados:")
print("- dim_tiempo.csv", dim_tiempo.shape)
print("- dim_cliente.csv", dim_cliente.shape)
print("- dim_geografia_cliente.csv", dim_geografia_cliente.shape)
print("- dim_producto.csv", dim_producto.shape)
print("- dim_vendedor.csv", dim_vendedor.shape)
print("- dim_pago.csv", dim_pago.shape)
print("- fact_ventas.csv", fact_ventas.shape)
print("- tad_ventas.csv", tad_ventas.shape)
print("- fact_pedidos.csv", fact_pedidos.shape)
print("- tad_pedidos.csv", tad_pedidos.shape)

Archivos cargados correctamente:
orders: (99441, 9)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 11)
products: (32951, 9)
customers: (99441, 5)
geolocation: (19010, 5)
sellers: (3095, 4)
category_translation: (71, 2)

ETL finalizado correctamente.
Archivos generados:
- dim_tiempo.csv (634, 10)
- dim_cliente.csv (99441, 5)
- dim_geografia_cliente.csv (14994, 5)
- dim_producto.csv (32951, 11)
- dim_vendedor.csv (3095, 8)
- dim_pago.csv (28, 3)
- fact_ventas.csv (112650, 21)
- tad_ventas.csv (112650, 55)
- fact_pedidos.csv (99441, 17)
- tad_pedidos.csv (99441, 42)


In [ ]:
print(fact_ventas.head())
print(tad_ventas.head())

print(fact_ventas.isnull().sum().sort_values(ascending=False).head(20))
print(tad_ventas.isnull().sum().sort_values(ascending=False).head(20))

print(fact_ventas["order_id"].nunique())
print(tad_ventas["order_id"].nunique())

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

                        customer_id fecha_key  pago_key order_status   price  \
0  3ce436f183e68e07877b285a838db11a  20170913       4.0    delivered   58.90   
1  f6dd3ec061db4e3987629fe6b26e5cce  20170426       5.0    delivered  239.90

#Validaciones


##General de formas

In [ ]:

print("dim_tiempo:", dim_tiempo.shape)
print("dim_cliente:", dim_cliente.shape)
print("dim_geografia_cliente:", dim_geografia_cliente.shape)
print("dim_producto:", dim_producto.shape)
print("dim_vendedor:", dim_vendedor.shape)
print("dim_pago:", dim_pago.shape)
print("fact_ventas:", fact_ventas.shape)
print("tad_ventas:", tad_ventas.shape)

dim_tiempo: (634, 10)
dim_cliente: (99441, 5)
dim_geografia_cliente: (14994, 5)
dim_producto: (32951, 11)
dim_vendedor: (3095, 8)
dim_pago: (28, 3)
fact_ventas: (112650, 21)
tad_ventas: (112650, 55)


##Unicidad de en dimensiones
Verificamos si las dimensiones si tiene una llave primaria, todos son true así que si lo tienen.

In [ ]:
print("dim_tiempo fecha_key única:", dim_tiempo["fecha_key"].is_unique)
print("dim_cliente customer_id único:", dim_cliente["customer_id"].is_unique)
print("dim_producto product_id único:", dim_producto["product_id"].is_unique)
print("dim_vendedor seller_id único:", dim_vendedor["seller_id"].is_unique)
print("dim_pago pago_key único:", dim_pago["pago_key"].is_unique)
print("dim_geografia_cliente zip_code_prefix único:", dim_geografia_cliente["zip_code_prefix"].is_unique)

dim_tiempo fecha_key única: True
dim_cliente customer_id único: True
dim_producto product_id único: True
dim_vendedor seller_id único: True
dim_pago pago_key único: True
dim_geografia_cliente zip_code_prefix único: True


##Granularidad de FACT_VENTAS
No debe de haber duplicados

In [ ]:
print("¿order_id + order_item_id es único en fact_ventas?")
print(not fact_ventas.duplicated(["order_id", "order_item_id"]).any())

print("Filas duplicadas exactas en fact_ventas:")
print(fact_ventas.duplicated().sum())

¿order_id + order_item_id es único en fact_ventas?
True
Filas duplicadas exactas en fact_ventas:
0


##Integridad referencial
Es donde comprobamos que todas las llaves de fact existan en las dimensiones. Esperamos de preferencia un resultado igual a uno o muy cercano a uno.

In [ ]:
print("customer_id de fact existe en dim_cliente:",
      fact_ventas["customer_id"].isin(dim_cliente["customer_id"]).mean())

print("product_id de fact existe en dim_producto:",
      fact_ventas["product_id"].isin(dim_producto["product_id"]).mean())

print("seller_id de fact existe en dim_vendedor:",
      fact_ventas["seller_id"].isin(dim_vendedor["seller_id"]).mean())

print("fecha_key de fact existe en dim_tiempo:",
      fact_ventas["fecha_key"].isin(dim_tiempo["fecha_key"]).mean())

print("pago_key de fact existe en dim_pago:",
      fact_ventas["pago_key"].isin(dim_pago["pago_key"]).mean())

print("zip de fact existe en dim_geografia_cliente:",
      fact_ventas["customer_zip_code_prefix"].isin(dim_geografia_cliente["zip_code_prefix"]).mean())

customer_id de fact existe en dim_cliente: 1.0
product_id de fact existe en dim_producto: 1.0
seller_id de fact existe en dim_vendedor: 1.0
fecha_key de fact existe en dim_tiempo: 1.0
pago_key de fact existe en dim_pago: 0.9991478029294274
zip de fact existe en dim_geografia_cliente: 1.0


##Nulos críticos.


In [ ]:
campos_criticos = [
    "order_id",
    "order_item_id",
    "customer_id",
    "product_id",
    "seller_id",
    "fecha_key",
    "price",
    "freight_value",
    "total_item_value"
]

print(fact_ventas[campos_criticos].isnull().sum())

order_id            0
order_item_id       0
customer_id         0
product_id          0
seller_id           0
fecha_key           0
price               0
freight_value       0
total_item_value    0
dtype: int64


##Métricas

Verificamos que los montos no se hayan inflado.

In [ ]:
print("Suma total de price en items:")
print(items["price"].sum())

print("Suma total de price en fact_ventas:")
print(fact_ventas["price"].sum())

print("Suma total de freight_value en items:")
print(items["freight_value"].sum())

print("Suma total de freight_value en fact_ventas:")
print(fact_ventas["freight_value"].sum())

print("Suma total de total_item_value en fact_ventas:")
print(fact_ventas["total_item_value"].sum())

print("Suma total de payment_value en payments:")
print(payments["payment_value"].sum())

print("Suma total de payment_value_item en fact_ventas:")
print(fact_ventas["payment_value_item"].sum())

Suma total de price en items:
13591643.700000003
Suma total de price en fact_ventas:
13591643.700000003
Suma total de freight_value en items:
2251909.54
Suma total de freight_value en fact_ventas:
2251909.54
Suma total de total_item_value en fact_ventas:
15843553.24
Suma total de payment_value en payments:
16008872.12
Suma total de payment_value_item en fact_ventas:
15846280.17



Notemos que

price en items = price en fact_ventas

freight_value en items = freight_value en fact_ventas

total_item_value = price + freight_value

Todos cuadran entonces eso nos garantiza la granularidad de fact.

Pero tenemos que la suma de pagos en la tabla de hechos no coincide con la suma total de la tabla original de pagos porque la tabla de hechos fue construida a nivel item de pedido, por lo que únicamente incorpora pedidos presentes en order_items. En consecuencia, pagos asociados a pedidos fuera de dicha intersección no forman parte del cubo analítico.


Para esto solo vamos a hacer la suma del total de pagos solo para pedidos que existen en items.

In [ ]:
# Total de pagos solo para pedidos que sí existen en items
payments_en_items = payments.loc[payments["order_id"].isin(items["order_id"])]

print("Suma total de payment_value en payments (solo pedidos con items):")
print(payments_en_items["payment_value"].sum())

print("Suma total de payment_value_item en fact_ventas:")
print(fact_ventas["payment_value_item"].sum())

Suma total de payment_value en payments (solo pedidos con items):
15846280.17
Suma total de payment_value_item en fact_ventas:
15846280.17


Se validó la consistencia de las métricas monetarias entre las tablas fuente y la tabla de hechos. En particular, el valor de pago fue distribuido proporcionalmente por ítem de pedido para respetar la granularidad del cubo. La validación mostró que la suma de payment_value_item en la tabla de hechos coincide exactamente con la suma de pagos de los pedidos que cuentan con ítems asociados, confirmando que no existen duplicaciones por el proceso de integración.

##Cobertura necesaria para el dahsboard.


In [ ]:
print("tad_ventas:", tad_ventas.shape)
print("tad_pedidos:", tad_pedidos.shape)

print("\n¿order_id + order_item_id es único en tad_ventas?")
print(not tad_ventas.duplicated(["order_id", "order_item_id"]).any())

print("\n¿order_id es único en tad_pedidos?")
print(tad_pedidos["order_id"].is_unique)

print("\nNulos principales en tad_ventas:")
print(tad_ventas[[
    "order_id", "order_item_id", "customer_id", "product_id", "seller_id"
]].isnull().sum())

print("\nNulos principales en tad_pedidos:")
print(tad_pedidos[[
    "order_id", "customer_id", "order_status", "tiene_items"
]].isnull().sum())

print("\nDistribución de tiene_items en tad_pedidos:")
print(tad_pedidos["tiene_items"].value_counts(dropna=False))

print("\nTop order_status en tad_pedidos:")
print(tad_pedidos["order_status"].value_counts(dropna=False))

tad_ventas: (112650, 55)
tad_pedidos: (99441, 42)

¿order_id + order_item_id es único en tad_ventas?
True

¿order_id es único en tad_pedidos?
True

Nulos principales en tad_ventas:
order_id         0
order_item_id    0
customer_id      0
product_id       0
seller_id        0
dtype: int64

Nulos principales en tad_pedidos:
order_id        0
customer_id     0
order_status    0
tiene_items     0
dtype: int64

Distribución de tiene_items en tad_pedidos:
tiene_items
1    98666
0      775
Name: count, dtype: int64

Top order_status en tad_pedidos:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## Conclusión de validación

A partir de las pruebas realizadas se concluye que:

- las dimensiones cuentan con llaves únicas,
- la granularidad de la tabla de hechos se conserva correctamente,
- no existen inflaciones en las métricas monetarias,
- las relaciones entre hechos y dimensiones presentan alta integridad,
- los nulos críticos son mínimos o controlados,
- y las tablas planas tad_ventas y tad_pedidos son aptas para su consumo en dashboard.

Por lo tanto, la capa analítica se considera validada para su carga en BigQuery y posterior visualización en Looker Studio.

#Subir sabanas a bigquery


In [ ]:
!pip -q install google-cloud-bigquery pandas-gbq pyarrow db-dtypes

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving smart_supply_chain.json to smart_supply_chain.json


In [ ]:
import json


JSON_INFO_PATH = "smart_supply_chain.json"

with open(JSON_INFO_PATH, "r", encoding="utf-8") as f:
    cred_info = json.load(f)

project_id = cred_info["proyecto"]
dataset_id = cred_info["dataset"]

print("Proyecto detectado:", project_id)
print("Dataset detectado:", dataset_id)


if "ejemplo_python" in cred_info:
    print("\nSe detectó el campo 'ejemplo_python'.")

Proyecto detectado: mineria-datos-493000
Dataset detectado: smart_supply_chain

Se detectó el campo 'ejemplo_python'.


In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=project_id)

print("Cliente BigQuery creado correctamente.")

Cliente BigQuery creado correctamente.


In [ ]:
datasets = [ds.dataset_id for ds in client.list_datasets()]
print("Datasets visibles en el proyecto:")
for ds in datasets:
    print("-", ds)

if dataset_id not in datasets:
    raise ValueError(f"El dataset '{dataset_id}' no existe o no es visible con tu cuenta.")

print(f"\nDataset '{dataset_id}' verificado correctamente.")

Datasets visibles en el proyecto:
- smart_supply_chain

Dataset 'smart_supply_chain' verificado correctamente.


In [ ]:
print("tad_ventas:", tad_ventas.shape)
print("tad_pedidos:", tad_pedidos.shape)

print("\n¿order_id + order_item_id es único en tad_ventas?")
print(not tad_ventas.duplicated(["order_id", "order_item_id"]).any())

print("\n¿order_id es único en tad_pedidos?")
print(tad_pedidos["order_id"].is_unique)

tad_ventas: (112650, 55)
tad_pedidos: (99441, 42)

¿order_id + order_item_id es único en tad_ventas?
True

¿order_id es único en tad_pedidos?
True


In [ ]:
#Limpieza antes de entrar a bigquery
import pandas as pd

tad_ventas_bq = tad_ventas.copy()
tad_pedidos_bq = tad_pedidos.copy()

for df in [tad_ventas_bq, tad_pedidos_bq]:
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if df[col].dtype == "object":
            df[col] = df[col].astype("string")

print("Copias para BigQuery preparadas correctamente.")

Copias para BigQuery preparadas correctamente.


##Función para pasar de datagrames a bigquery

In [ ]:
def subir_dataframe_a_bigquery(df, nombre_tabla, write_disposition="WRITE_TRUNCATE"):
    table_id = f"{project_id}.{dataset_id}.{nombre_tabla}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition,
        autodetect=True
    )

    job = client.load_table_from_dataframe(
        df,
        table_id,
        job_config=job_config
    )

    job.result()

    tabla = client.get_table(table_id)
    print(f"\nTabla cargada: {table_id}")
    print(f"Filas: {tabla.num_rows}")
    print(f"Columnas: {len(tabla.schema)}")

In [ ]:
subir_dataframe_a_bigquery(tad_pedidos_bq, "tad_pedidos")
subir_dataframe_a_bigquery(tad_ventas_bq, "tad_ventas")


Tabla cargada: mineria-datos-493000.smart_supply_chain.tad_pedidos
Filas: 99441
Columnas: 42

Tabla cargada: mineria-datos-493000.smart_supply_chain.tad_ventas
Filas: 112650
Columnas: 55
